# ToolStrategy

## Schema

### Pydantic

In [44]:
from typing import Literal

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from pydantic import BaseModel, Field
from rich import print as rprint

load_dotenv(override=True)
model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", timeout=30_000, max_retries=1)


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 定义Pydantic Schema
class CustomerAnalysis(BaseModel):
    """客户分析报告"""
    customer_name: str | None = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] | None = Field("潜在客户",
                                                                                         description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str | None = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] | None = Field(None, description="消费水平")
    send_email: bool | None = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
请分析指定客户的情况：
1. 先搜索客户数据库了解最新情况
2. 如果是VIP客户，则发送感谢邮件
3. 基于搜索结果生成结构化分析报告
4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件

注意如下约束：
1. 每个客户最多调用一次 search_customer_database。
2. 如果搜索结果包含“无记录”，禁止再次调用搜索工具。
3. 此时必须立即调用 CustomerAnalysis 生成最终响应。
4. CustomerAnalysis 是结束智能体执行的最终工具。
"""),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户张三 & 李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    rprint(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='8440deaf-4bdd-4c50-a6e7-98ee42fd510c'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785169924-83t9hr0csxgblxLcyRHl',
                'created': 1785169924,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa46b-47ae-7041-9c21-943ff6df1407-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_t8XtgzkX9xPLqmKISJ8gi5Dc',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 361,
                'output_tokens': 20,
                'total_tokens': 381,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='0bba26db-0d72-4d3e-939a-460f5a839fd1',
            tool_call_id='call_t8XtgzkX9xPLqmKISJ8gi5Dc'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': "**Processing customer analysis**\n\nI see that I need to send an email after 
getting the VIP result and then move on to the final Customer Analysis. This order is important because of certain 
conditions that need to be met. So, I'll ensure to call the email function first before proceeding with the 
customer analysis to keep everything on track. It’s all about following the sequence to make sure nothing gets 
missed!",
                'reasoning_details': [
                    {
                        'type': 'reasoning.summary',
                        'summary': "**Processing customer analysis**\n\nI see that I need to send an email after 
getting the VIP result and then move on to the final Customer Analysis. This order is important because of certain 
conditions that need to be met. So, I'll ensure to call the email function first before proceeding with the 
customer analysis to keep everything on track. It’s all about following the sequence to make sure nothing gets 
missed!",
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    },
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqZ4gKtHQBJhk9g793k9WKao8L6rPbsP5uzkXedaBaSTMbtYpRa-TXnIZHu9AdRHc8DOGynDUkAoV7TLfmvt8XEpc0Z4T7B_wMoqC1UShXbt
Fj-JAWZ2a7sum7evcRKj2nmZ4NkwV2FNP72cq6cDEQ036pIku_T720xPXPDGtabGPJ20NeMIauvRlZ-kwPtUF1o-EakLOyyJXpidR1pZwxQsIPZ7N-W
cE6CNbSLI3DLvUY6GVMD9sMwLFeQx0CDKhOV377OlOzrb9SA6-5lgbqWE5Lwj3xBH4-FAkULWk0aeerJ6lKBqZyng7sMuw-TfDNejgTM2VnKMgz4Ej0
UOYuQhYtx1GkYPHHytvZvloyb2Kjy7opW1JYJBjI3i0Crsqr9bGe30-b7VF-c_V4eFcxgu8h8LqOtPP1uPeHAT36jx3DQTEPafMBogGKbEiqQwgJXwU
A1NWlcXBfXktAEA3-Wnc3A3IFluAWDShCb9wpldSSPbXfPknUyyOkzqNQ3nf5Gh7JalW4lMSbhHxO0M9xi9AJULyJ55EsfVjQNV3SXj-HlzN7uFcKVZ
V0jgjXKtrS09Cv0EuiJyAxJVUpT-l4GXaauzQuNQFcdjh1u799bPyENiQgt5gvBr1S3JaX_8W63__SEbnLUKAbKTSOX1Y5ZEUkNvUrHbNwq0ifnQ870
6L4HruWTI67MRUyh2OaLywczxTny610266X2QGAONm4JXKJ6MvZmD2Jcf56L7JMwAUf1--IfQoQV5prZe3lcKKDuBS-1c9ao5kzFVxe-F8HB16jhjLH
u-oiH0pSLMpXwEQ1yc9M2SAwsACdTZ3oIH6GL2TkdYvV1M1ijYva8d9Lec53Z94G1EK0VpdFwKqv1z635_iEXnG17BriicHVDEM0EcWGdneyfkO78su
HxtF9L4gCvjViHftKXpwDomYlzALkd8hFhiWpOeH8tV32gfTEzBjNJLwf8CtGECqtPBaV4LIdnv4vTX50YqHy7skX2fZhHEHHoPrXDqFT6a9CrdNwsE
Y6nZP9E2tehnw4Gnykwy1IsH7A

CustomerAnalysis(
    customer_name='张三',
    customer_tier='VIP客户',
    recent_activity='最近购买日期：2026-01-15；累计消费：$15,000',
    spending_level='高',
    send_email=True
)

### TypedDict类型

In [46]:
from typing import Literal, Annotated, Optional, TypedDict
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)
model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", timeout=30_000, max_retries=1)


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用 TypedDict 定义客户分析报告 Schema
class CustomerAnalysis(TypedDict):
    """客户分析报告"""
    customer_name: Annotated[Optional[str], None, "客户姓名"]
    customer_tier: Annotated[Literal["潜在客户", "普通客户", "VIP客户", "流失风险"], "潜在客户", "客户等级"]
    recent_activity: Annotated[Optional[str], None, "最近活动"]
    spending_level: Annotated[Optional[Literal["低", "中", "高"]], None, "消费水平"]
    send_email: Annotated[bool, False, "是否已发送感谢邮件"]


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
请分析指定客户的情况：
1. 先搜索客户数据库了解最新情况
2. 如果是VIP客户，则发送感谢邮件
3. 基于搜索结果生成结构化分析报告
4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件

注意如下约束：
1. 每个客户最多调用一次 search_customer_database。
2. 如果搜索结果包含“无记录”，禁止再次调用搜索工具。
3. 此时必须立即调用 CustomerAnalysis 生成最终响应。
4. CustomerAnalysis 是结束智能体执行的最终工具。
"""),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user", "content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户张三 & 李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    rprint(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='1587ab6f-6220-4fa7-a6ef-dc6ba6257bf9'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785171374-gkAaAumHvm69yogfJLYB',
                'created': 1785171374,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa481-6fa5-7cb2-ae41-cca01887ac33-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_ZYgJlRteJ8SkR22BRG4PwQ3c',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 304,
                'output_tokens': 20,
                'total_tokens': 324,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200',
            name='search_customer_database',
            id='6f52ba00-c3bd-439f-84e4-43e40e9e7e45',
            tool_call_id='call_ZYgJlRteJ8SkR22BRG4PwQ3c'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': '**Organizing customer analysis**\n\nI need to call for a customer analysis to
determine if spending $3,200 is considered high, especially if there\'s no specific threshold. It\'s important to 
have a structured report for clarity. I\'m just wondering if labeling it as "high" might be misleading; I want to 
ensure I\'m sending the right information. Let\'s see what we can uncover about their recent spending habits. This 
will help in painting an accurate picture.',
                'reasoning_details': [
                    {
                        'type': 'reasoning.summary',
                        'summary': '**Organizing customer analysis**\n\nI need to call for a customer analysis to 
determine if spending $3,200 is considered high, especially if there\'s no specific threshold. It\'s important to 
have a structured report for clarity. I\'m just wondering if labeling it as "high" might be misleading; I want to 
ensure I\'m sending the right information. Let\'s see what we can uncover about their recent spending habits. This 
will help in painting an accurate picture.',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    },
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqZ42y2J9s0nMmB-YO3ElgPixuWtBXJddpXHIhCZziChIpRDPiMiDH0RHshCSKj1mOC2gro81j1FpvS7u_cAODojen5wTg0IVFiu9VX0WE_1
gC1nB9h1m0fL3O0G5o4VJojGNBrbTFHVtquBSrymj8c2BePwLx7_ax8c-0kPSjD2lG3chtWtGWhGYIcJRtNIeeDDbT9d4uwas50ckWeVAVYoJBQ7yxS
WCIwMYJUG3NDDwyTkW9eBrjYV-6giWm_bUx2tqmja3LCpQOJnxQHQqr3Xn1YlsrrgfF3J8letaIyRprYXY8CiStMOJECrIWSU8LXi3WHuR8p_bQ2Dl2
BxfiEplIC9wyo-dXJB94v_4Vilgmya8Z59bxA-3E078kX_YPTtmHpEt1gC0fCgBHRvSSxmnY1UCEV7PY3l-0cMC82XOjpLqhJJ1VorPuhPsF_HrqjIf
dXdsdO9op_S0OfP6KNiv2OJvRT6jwHC3pry3ZHIq_05nNLCdWNe_0epPZoRAou09AhhizlKLX74ZjWBXRQXv73TAyI9GiS68-lqHKwDwZkEB2bKKnFC
N09a2xfg4pwUW-iwlgtfl_IrPPhSzOF6KKivWy_dGoFMNrqwOcPW6kVFqqL8EbaaAYVzgsw1mvcFOkqEiMZrCsBfDVKmY2rKWNH7xJ2qCA2QRfWmei_
FG3xlEtz2gorCEssWwbGL0b2ZH_s3QCpeaJuRbEsqWlEEtsG1CGPcSXkR4QdPi7zxuo5iIvtkFCQ7Si_PuiVGB_d3mohsfJeYCSqmnvaO3BZtgNF4yy
LXDxY5pR7ipdpMQuy0XM9oNrYRW6w_w5xDKyU7i3vxGvBeON2LqUBUIdmEV1K4Yj9CN3sjr2EufgnjTZecQniHemmvn3Ko3zpqCi5Lt0Chtgn074PBH
8Dy437jwO-WcR4Z2L_YekouE-9PkEEGCcR16GU78y3V5qRx9IJXrRJY

{
    'customer_name': '李四',
    'customer_tier': '普通客户',
    'recent_activity': '最近购买日期：2025-12-20；累计消费：$3,200',
    'spending_level': '中',
    'send_email': False
}

### JsonSchema类型

In [51]:

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from rich import print as rprint

load_dotenv(override=True)
model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", timeout=30_000, max_retries=1)


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"

    # 定义 JSON Schema 替代 Pydantic 模型


CustomerAnalysis = {"title": "CustomerAnalysis",
                    "type": "object",
                    "description": "客户分析报告",
                    "properties": {
                        "customer_name": {
                            "type": "string",
                            "default": "",
                            "description": "客户姓名"
                        },
                        "customer_tier": {
                            "type": "string",
                            "enum": ["潜在客户", "普通客户", "VIP客户", "流失风险"],
                            "default": "潜在客户",
                            "description": "客户等级"
                        },
                        "recent_activity": {
                            "type": "string",
                            "default": "",
                            "description": "最近活动"
                        },
                        "spending_level": {
                            "type": "string",
                            "enum": ["低", "中", "高"],
                            "default": "低",
                            "description": "消费水平"
                        },
                        "send_email": {
                            "type": "boolean",
                            "default": False,
                            "description": "是否已发送感谢邮件"
                        }
                    },
                    # 所有字段都是必须输出的
                    "required": ["customer_name", "customer_tier", "recent_activity",
                                 "spending_level", "send_email"]
                    }

# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
请分析指定客户的情况：
1. 先搜索客户数据库了解最新情况
2. 如果是VIP客户，则发送感谢邮件
3. 基于搜索结果生成结构化分析报告
4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件

注意如下约束：
1. 每个客户最多调用一次 search_customer_database。
2. 如果搜索结果包含“无记录”，禁止再次调用搜索工具。
3. 此时必须立即调用 CustomerAnalysis 生成最终响应。
4. CustomerAnalysis 是结束智能体执行的最终工具。
"""),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    # "messages": [{"role": "user", "content": "请分析客户张三"}]
    "messages": [{"role": "user", "content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户张三 & 李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    rprint(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户李四',
            additional_kwargs={},
            response_metadata={},
            id='3f384a47-ae89-42e6-84c4-754ef6fca643'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqZ458wd5QXgwdxknLjyVexY4jOZgTgTG8cuhXwp27Px4J5AreMD542dMy4tRYfbScAZ29DXY3KIdH6CfTpGH9JvAhubhwnloAFKbaFcOMvE
V0urq4rK1FILytNZM_bTgHTiLlluzQLDNLkU8-jXcOfsn1Wyc9scaw_IVHehVK2nte5dKP62FFsXkvYNC_RSzQMqitWSHvDev12a3mxM5CJmXNOJSXn
3q8oa2mWaiaiH2OMKE6xwYXw6_hNjfEDKM0cK-gYfvEJKPZu9fApEovA8TBEL_2eHg_2kTs36wcjHr-K9Tx4TimG1VwvRXap_sEfDyD2lVgPIyfsXd0
hJXnFbvnbgyJ8Tw292e2AFs22ZRUObXy63IyiNkIoNM0Uy59UqtNexQkUCua6x0ccZDlGWAwnERaXicvWC9DVa6Vz4gZVDDvTxdmHbOc-fnptd0364a
Zt9N2b-w3Tqgi1bjqaLN5tklgmHiQTnrih1NB9_pHu8hF9qtUZDpXe1eYhOpFgPM8C7JU9Or7smZ25dPhD1CGCaD6RtRYo3BWWnrFDmen_fllpfww9T
ix8jKQEcs0ezJaxJ8d0PWLIYSHduwS3voEylGgFI6237WKX5qKML-yHTZ55X8QLAhkWJRjmwlXl_-R1PC4MZoy7v8u99fozQWPqsSKCgteh73e_--Sg
OItXxb7xJ61SjJfgAMEa-PXACQC80jEjNBsaf6OFlztTyT98JyOQSf2lS6taBJejr6d6IS4PpxP7v-ynSaC_jCd07Om7MYJ0cZodrlhqSOot_sYjDoc
0YUSZeHCRmWTRLYad3dT8E4YaDd6Hw1u7Y1IoRFmGCKZkaie3znEsEKv_18zw6PdPmAWkBVhe0Ah03zVSsOVq7VoXuMzCMizWTzaDUIAsVJiCI5yItE
k0f0khKMYMudIj0ovWcnYcV8A-cyiacXNAJdZ7dMf_9Q51i0IuFUc',
                        'id': 'rs_08dc7ab91ef4e154016a678e7bc55081a1b8ad71c557b20f5c',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785171578-dZuF8mq033s45avfL4e7',
                'created': 1785171578,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa484-8f9d-7091-befb-0632a929acdf-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '李四'},
                    'id': 'call_LsI6gkL5e8AJ8yUjZRctfXYU',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 349,
                'output_tokens': 31,
                'total_tokens': 380,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 9}
            }
        ),
        ToolMessage(
            content='客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200',
            name='search_customer_database',
            id='37712f4b-e62b-4203-9a0b-8b679d708e64',
            tool_call_id='call_LsI6gkL5e8AJ8yUjZRctfXYU'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': "**Checking customer spending**\n\nI need to prepare a report, but it can’t be
just an ordinary email. I think I should call the CustomerAnalysis function to get the latest details. I'm noticing
that spending seems high, around $3,200, which is probably over the typical range. I’ll check recent purchase 
activity to find the exact date so I can provide accurate information in the report. Let’s make sure everything 
aligns well before finalizing it!",
                'reasoning_details': [
                    {
                        'type': 'reasoning.summary',
                        'summary': "**Checking customer spending**\n\nI need to prepare a report, but it can’t be 
just an ordinary email. I think I should call the CustomerAnalysis function to get the latest details. I'm noticing
that spending seems high, around $3,200, which is probably over the typical range. I’ll check recent purchase 
activity to find the e

{
    'customer_name': '李四',
    'customer_tier': '普通客户',
    'recent_activity': '最近购买日期：2025-12-20',
    'spending_level': '高',
    'send_email': False
}

### @dataclass类型

In [52]:
from typing import Literal

from dataclasses import dataclass
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage
from langchain_core.tools import tool
from pydantic import Field
from rich import print as rprint

load_dotenv(override=True)
model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", timeout=30_000, max_retries=1)


# 定义工具
@tool(parse_docstring=True)
def search_customer_database(query: str) -> str:
    """
    在客户数据库中搜索信息

    Args:
        query (str): 客户查询字符串，例如 "张三" 或 "李四"

    Returns:
        str: 客户记录字符串，包含客户姓名、等级、最近购买日期和累计消费
    """
    # 模拟数据库查询结果
    if "张三" in query.lower():
        return "客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000"
    elif "李四" in query.lower():
        return "客户记录：李四，普通客户，最近购买日期：2025-12-20，累计消费：$3,200"
    else:
        return f"关于客户{query}，无记录"


@tool(parse_docstring=True)
def send_email(customer: str) -> str:
    """
    发送感谢邮件

    Args:
        customer (str): 客户名称，例如 "张三" 或 "李四"

    Returns:
        str: 确认消息，包含已发送的客户名称
    """
    return f"已向 {customer} 发送感谢邮件"


# 使用Dataclass定义Schema
@dataclass
class CustomerAnalysis:
    """客户分析报告"""
    customer_name: str | None = Field(None, description="客户姓名")
    customer_tier: Literal["潜在客户", "普通客户", "VIP客户", "流失风险"] | None = Field("潜在客户",
                                                                                         description="客户等级,只能是潜在客户、普通客户、VIP客户或流失风险")
    recent_activity: str | None = Field(None, description="最近活动")
    spending_level: Literal["低", "中", "高"] | None = Field(None, description="消费水平")
    send_email: bool | None = Field(False, description="是否已发送感谢邮件")


# 创建智能体
agent = create_agent(
    model=model,
    system_prompt=SystemMessage(content="""
请分析指定客户的情况：
1. 先搜索客户数据库了解最新情况
2. 如果是VIP客户，则发送感谢邮件
3. 基于搜索结果生成结构化分析报告
4. 如果用户提问与客户记录无关或找不到客户信息，则返回空对象，不发送感谢邮件

注意如下约束：
1. 每个客户最多调用一次 search_customer_database。
2. 如果搜索结果包含“无记录”，禁止再次调用搜索工具。
3. 此时必须立即调用 CustomerAnalysis 生成最终响应。
4. CustomerAnalysis 是结束智能体执行的最终工具。
"""),
    tools=[search_customer_database, send_email],
    response_format=ToolStrategy(CustomerAnalysis)
)

# 执行分析
result = agent.invoke({
    "messages": [{"role": "user", "content": "请分析客户张三"}]
    # "messages": [{"role": "user","content": "请分析客户李四"}]
    # "messages": [{"role": "user","content": "请分析客户张三 & 李四"}]
    # "messages": [{"role": "user", "content": "请分析客户王五"}]
    # "messages": [{"role": "user","content": "今天天气如何"}]
})

# 处理结果
rprint(result)
if "structured_response" in result:
    analysis = result["structured_response"]
    rprint(analysis)

{
    'messages': [
        HumanMessage(
            content='请分析客户张三',
            additional_kwargs={},
            response_metadata={},
            id='bf045b1d-7b1e-4e37-b9ef-1b44c0dbd08f'
        ),
        AIMessage(
            content='',
            additional_kwargs={},
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785171689-mQcryT4hzJGNnDwtnjiG',
                'created': 1785171689,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa486-3d03-7bd3-acf1-4c1887fdb555-0',
            tool_calls=[
                {
                    'name': 'search_customer_database',
                    'args': {'query': '张三'},
                    'id': 'call_h4pYyVWU4S0cfewUKddMYrOr',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 361,
                'output_tokens': 20,
                'total_tokens': 381,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 0}
            }
        ),
        ToolMessage(
            content='客户记录：张三，VIP客户，最近购买日期：2026-01-15，累计消费：$15,000',
            name='search_customer_database',
            id='b4f45026-015d-4d4c-8cd8-908371cd9995',
            tool_call_id='call_h4pYyVWU4S0cfewUKddMYrOr'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqZ47sum5m6kq2SKVhFi5x1ta-y0hXXdkEe2kKQR2twmDW4yoLgPJe4UB-OYSMmwRmZyPvJVcv0YGpxOtwMVaA-4kVciNMbkP32yKt39cXhe
3pxud4lFDPjEO652wdadRbKATYBVF54oUdQ6C5qliR3V5iCrNrVYAl8p01Px3f00NRbKyhbIoVnhL78rdUaubVm8Yte6_vsgPyARW1SnJT_ucws3pHx
SlMm6udhMx6OgsLF_BA-yIoWSh5XEf5C7KZmlTqDbwv8aIt22Oin4SP6JIrI3eYxoqf4JezbFyXvOYlc4-YMBHfZJ9NhhMsHFH5i3LuZnjwQE5mMD03
QPxW1N5McBJgtPxAwvm30r5Tp91urQbcGNNRjsk-UDnnQYwcEllsxW6uam2wE4K40ZcrIVr5rRvd2tDDEHfd-E66oB_4OLW5GczBR6M4FgMbQ_t-G1m
LRYHqNoA2tlQoO_wD97fSu6LQbdO1B7FIbbsSprl3JUpympaGkIh_F8_2golcR2zIWblNtIg6UZJESJQVcUpPRRnv4pUyhnGgFzV_heZaIDpb4NbtHG
FA9pUFY1dzXCnxD86zPl_PHO7pv9iCzdbKJWOAi5GOLPMvPshFJtZjvyLQlX9JDpXw2VFTuULxRK5I92eeQvRJL4wjsv2FoExLwzgPWPtYpiJIJdSNO
VPHHQKMlX9HNN8kJhSHxE-EPL1G4B9cuXcZrkGaTrtLBok9VYBFZgyAoxNMzZ4duu7jJGE-Gvumfu6UkOpyETeB4iS6LphfrJBZrPd9569LxnXeQt6Q
EvhUDYLSCiykUKuo9hVNtxn3KbFkWg_QguSmMYjZ2IShAYbEvTv0IshQQiOlEKchBpipCJBKMzaNqB42pEKy5yIVaOlLjyvedHFKBhu4rtsFqNW3eYa
LNhC61WqXEmZkD2ZEi6qy89KeQjkW1K9e4rxHJosZ-64FQHmNVIlY9DOFe1K7xh-Z7zQCRrSfVpzJbT243_Xo42GnDjqV3raafKlZv-KLYxE4gEvHqq
bWrZGkbMJ5MOyOX8_9z3B_UbwbaORT0qAgZ4yWAeflvq4=',
                        'id': 'rs_000e26145fc6b0e4016a678eeb59a0819f88369b22eb8f1941',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785171690-jiH7IZYIqD9TJbissFC8',
                'created': 1785171690,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa486-445a-7331-892c-81bf99162e67-0',
            tool_calls=[
                {
                    'name': 'send_email',
                    'args': {'customer': '张三'},
                    'id': 'call_rBsj53oCHKskMxJJ2axJZoSE',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 420,
                'output_tokens': 43,
                

CustomerAnalysis(
    customer_name='张三',
    customer_tier='VIP客户',
    recent_activity='最近购买日期：2026-01-15；累计消费：$15,000',
    spending_level='高',
    send_email=True
)

## 多schema联合模式

In [53]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.messages import HumanMessage


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


class EventInfo(BaseModel):
    """事件详情"""
    event_name: str = Field(description="事件名称")
    date: str = Field(description="事件发生日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventInfo]
    )
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912")
        ]
    }
)

for msg in response["messages"]:
    msg.pretty_print()

print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：小明的邮箱地址为：shkstart@atguigu.com，手机号：12345678912
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_fp2lVr6I5M3ze3rWogYZZ80e)
 Call ID: call_fp2lVr6I5M3ze3rWogYZZ80e
  Args:
    name: 小明
    email: shkstart@atguigu.com
    phone: 12345678912
================================= Tool Message =================================
Name: ContactInfo

Returning structured response: name='小明' email='shkstart@atguigu.com' phone='12345678912'
name='小明' email='shkstart@atguigu.com' phone='12345678912'


In [54]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：2026年高考报名人数突破1200万")
        ]
    }
)
for msg in response["messages"]:
    msg.pretty_print()
print(response["structured_response"])

================================ Human Message =================================

从这段话中抽取结构化信息：2026年高考报名人数突破1200万
================================== Ai Message ==================================
Tool Calls:
  EventInfo (call_GbAoozMdMvnhP3TZkzM58ZfA)
 Call ID: call_GbAoozMdMvnhP3TZkzM58ZfA
  Args:
    event_name: 高考报名人数突破1200万
    date: 2026年
================================= Tool Message =================================
Name: EventInfo

Returning structured response: event_name='高考报名人数突破1200万' date='2026年'
event_name='高考报名人数突破1200万' date='2026年'


## Tool Message Content

In [55]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from pydantic import BaseModel, Field
from rich import print as rprint


class ContactInfo(BaseModel):
    """用户的联系方式"""
    name: str = Field(description="用户姓名")
    email: str = Field(description="用户邮箱地址")
    phone: str = Field(description="用户的手机号")


load_dotenv(override=True)
model = init_chat_model(model="openrouter:openai/gpt-5.6-luna", timeout=30_000, max_retries=1)

agent = create_agent(
    model=model,
    response_format=ToolStrategy(ContactInfo)
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='a99dbf94-d92a-4e4a-874a-f7131070cd61'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqaMu7ACVDz-ED0GduLEQvB9wnZlcmq-8NNf7VUcS3r4XiS1pnYHCrlw5rN6R53hgRnISwzusWMYtxHkZxHeCk9HBfkN1k2z6AQBrZrifttT
vdMnRxowVq0i6UFi-u9EQ1-OFRcHO7aGAPKKaPEIlYJE_hOHlycwZ99TXc8HIM3XpaNVR-kUCQuSSVVtMWN_Txus5Td8KIwa9GZCrKesmYLeWUUF00Y
ltmysAS8cOODP9L0SmMHx0jrfHNZBYh1tiW33EWHytUeBQQXaAz5B1D3PbuObqTWeMhrccAKMpVA4WUQBvpxuz7QYYNAcyuPT8yyq6_VGzY1FfLhwB8
SX2KGHbdoFo-3evqXL__JjwmQy0s7ILCpVwMqBNlU1BAG6tcocDa2dLOXyd5VntMMjPeKrw7GSovLpDRnDWtBuWLgt4g6613IFGMAGbJFg1YTtygHfu
atBtZonp5iteNsKCFy076QD4BKHI5k2jOs1pmGxa72ukRv4Z2ekrStLyYYVFD9Jzj-6Tl4Agg16WG9x5vs6R64V0D4gN5QePpepHIez6voCcXb-ylkP
67AyGrHmuAEG4gs5XmLZ7n6XTIdZVo2ZS-mLmY_fo7SmspkIFXl34mw17_530XjAnFMnNyThqNIrT9L8VY3f59hCmSojCcSlkQlPTydnazWAv0fuhvI
6kglKDaDeVBNfPVpE75355trxcLJjJWRZerLo1rYZgCD4COqi1facQiYRXJpYqc6srpaxtE51bouNQTrZ35rmWx7x9UmK1_Xe5y_Q-zVB8N-rX52MVA
3lkSt4c_YNJLz6VapQeQB3SQT96G-ZaKaOXcN1Q2HC-togiq3OZ3sbo7mxJ9xCinxZBNpybU0V87G6v8_tJuo_dcnr8jGT3qM09SS0nsOrClDsv9xZA
aG_Zt8SkyTqtzs2N-gY_qVyaQTRpHC-ed886pDMkxQW5pObw3jqMcsvDfDuVk1pq5ySxG2_PR8jGfNjiGDN-siLMO9BL103k=',
                        'id': 'rs_04de075fb947c19d016a68cbbb6c1881a384ed63aaf13f01ed',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785252794-CNJJ2YOl0BTDWvDFTm27',
                'created': 1785252794,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa95b-cf1b-7ba0-b769-c985bb2932a4-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_VZrif6rwBZgoT1DDqjsLrWYO',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 52,
                'total_tokens': 143,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 15}
            }
        ),
        ToolMessage(
            content="Returning structured response: name='小明' email='songhk@atguigu.com' phone='12345678912'",
            name='ContactInfo',
            id='a289fcd5-2b3f-4207-b7f7-6011d7fbc548',
            tool_call_id='call_VZrif6rwBZgoT1DDqjsLrWYO'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

In [56]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(schema=ContactInfo, tool_message_content="Structured Output")
)
response = agent.invoke(
    {
        "messages": [
            HumanMessage("从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912")
        ]
    }
)
rprint(response)

{
    'messages': [
        HumanMessage(
            content='从这段话中抽取结构化信息：小明的邮箱地址为：songhk@atguigu.com，手机号：12345678912',
            additional_kwargs={},
            response_metadata={},
            id='dc0d0df9-d13f-4058-8d83-a3dcf7ab1ab1'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqaMvEpGLeU-oBvrHF-MwpwTv8BRmN49Z2xSuuTn5AXdOnAnOfDlnKpeLy7E5iaPPxlabhb1pKhGpvMJtu2WJuPXrxT5YD1dz--H6C3nJUEB
8GcRy2nHFAFbiuMOVWkVYqgNhS0KJwy5ro61uCC24VrIhwYFtuZlgvcy3h9kSHEgrW3QduR4zHGNJZ0LsQjq5TFtHCqVSjXD9yqSzSPfLYj3Nh5IRPz
eXVxx1OtgRdrTdi659ESi1K5wlIKlhLvEObdffKlP7Xi6uo6WjRv7tRitZFeYEockK3nuBSx3V8vulYjAtSPjSy5sy04cqwrgl9fT1fYPSHxVsqrEZ2
ktX5sc-bFfqOZj521cyjEFW5muSsxmHu5XR8rqRjf0EJH_HmZwzRk4gJ5dQjL2otlgih1fanwb-AGA0ZmgwrfjvfG8P_sVtJ2edmMY1dVP0_qwRAyeE
YxzC_AZYoJakxGmdfwcuDXT94Ln5gwbJDxnbEXpv0E-Ig9SdCB0QJyQ5f4yeI5XVQ0T929W8tnbcYGSEDjwF3df3ECzgOkqgQKje-XgvmqetwbaV3x8
SSCd0cTSCymbNXp6ZRDna_Ar44Lx_aQNpBfYzRS6SDcyiefb1SlMGgNFLX9t1uCtZWeHMiDhVxwgP2Nk1CWGrBA_EiaZUMaE8LzwyP4An8a0ZG9G73-
RAWFKn0qPN_tmyKhRJK7o7WuWvAebl8YrDC-C8HUrktu0lT5v7GUAC9uXYFOdK6VWH234caRq7TfaDUtHpankFsXO-uZCAe3rdotppBOFqR-Z4uunfC
DWp8hV_5mJYDNMG9jPsxu-MNCnHhYlNRM8QM5ItwUmr3K9hWGBCMzkA6UgkKgefbOAVH11m8f-lfTCr0bELpLwtzfd3nREJcr3woc4joc72TT_ToMzZ
0kv8kY2Gk_-opEL8i23mClwUYdwp01dtfAKSQUIFpImn2hMFN8d0PE91Tc2UyY-17PctPQh8f--oog-G5GlOtReyhCqOL29E=',
                        'id': 'rs_03c46645ed6154c5016a68cbc411c0819f8c6703b59ae89cf3',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785252803-YO4AhjJbz8dZVNIfPNQZ',
                'created': 1785252803,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa95b-f203-7a33-951e-3ae6cd43355c-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '小明', 'email': 'songhk@atguigu.com', 'phone': '12345678912'},
                    'id': 'call_Hqu2vxrOYT7Izobozqbhbz7F',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 91,
                'output_tokens': 54,
                'total_tokens': 145,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 17}
            }
        ),
        ToolMessage(
            content='Structured Output',
            name='ContactInfo',
            id='d85230da-4b3d-4211-9947-9d7a9ff264b5',
            tool_call_id='call_Hqu2vxrOYT7Izobozqbhbz7F'
        )
    ],
    'structured_response': ContactInfo(name='小明', email='songhk@atguigu.com', phone='12345678912')
}

## handle_errors

### 设置为True/False/固定字符串

In [57]:
from pydantic import BaseModel, Field
from typing import Union
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy
from rich import print as rprint


class ContactInfo(BaseModel):
    """个人联系信息"""
    name: str = Field(description="姓名")
    email: str = Field(description="电子邮箱")


class EventDetails(BaseModel):
    """活动详情"""
    event_name: str = Field(description="活动名称")
    date: str = Field(description="活动日期")


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=True
        #handle_errors="请检查输入数据"
    )
)
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"}]
})
rprint(result)

# for msg in result["messages"]:
# msg.pretty_print()
#
# report_data = result["structured_response"]
# print(report_data)

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='60dcb918-f911-40dd-a873-ae884e395af7'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqaM12zawRF-_e2vq3azwDFTq1O8tjJcruvrMKH2ZdW3R-SCIB3Q7FeiDiDs-lnzeSl03x3XF0mALyU4w7uIVROg7b_6OKaor0G-ss_66yrT
uveTgfzJ8Dy0GGTvoZxrKCP9ePHUZFsOy-vaDswrBciq8nSE2tWYAIDF9ydizVfR9AEeSviB5i20MWa1jtlnDkg2dFWZZGKGiNJvaEwxYDDZ5PAvDVw
_Jmy8FmlHsKvoVdmTt7uyf2leYHIP2PagYE-kMyzKBxlGqLGonea-9RwdPHWtfv179pQpMrEECb1wqIBbc_zPcWngjmWS5SLFJ8wvlYktCKvlM_8M-K
n9X-8pmH4TBVnPG_3PFvgXKNBoH-a_KtYTVUFFgtaDX-yB2CBHMEfTvpDUb0JW5nKwx8xczubQK9G116j-t0bUNW7XF5jxyYqxTjaxZjiw9u3SqoBU3
5blx3xmAj8pxXZSOq-_bfejJuza_95l3qjEFgCtpJdkdwCFt_C9MdAeTN9QMww_T90kXFA4w_zsncuyvlVVnkEPmdeIfszW37JUpIINHlkHs9lN2cFD
dx2tbOV5ZpFYPe5sEzA6GsndCmcLEk-UczFZl0I3EtRh9gHoWDcjOibCwD_FucThhHStarLdNPMrNwQU0yj6CoECSBUSwor6XC7Fmg8VOzR1xt0WaAW
K9x-vWRCkbHiZMvPOCQB-22F4qoDALxRJhQnbIkG2MV-t-vlv7S8BxHeOwb2HwuYM47SKtxgEbK1dg1LtuKlYZNqAMPhAq5oXSK6ESr04ox1QgJJoLN
_9Y56ypPUC1upE5QWoAHj-bn1dqInKosVuuxkUmjoqTTMqa133QPu_IlcfFryYWS_ph31N5KskZmC8EKi-x-AUSqp-ENyHkJPAoM1eTQwEsAnRFllj7
OcHTYUrtvMdHnl-g9H3_vXdbVvwhvno3ExDpwAA5NS1pgD-1mMNoDAdgocm1Nw-xM3JTYLIi3EA==',
                        'id': 'rs_056266caf6506aff016a68cd76506481a0a3aa8ba370efed33',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785253237-9CAfGpv2heBaeqbiCvXn',
                'created': 1785253237,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa962-8fb4-7d40-811c-2aa428d690f5-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_xcg6VNPMQbSv3QWLy2pdGOpu',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_hG46LFbb0mFpsLF7u82v9fGB',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 126,
                'output_tokens': 87,
                'total_tokens': 213,
                'input_token_details': {'cache_read': 0, 'cache_creation': 0},
                'output_token_details': {'reasoning': 14}
            }
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='ContactInfo',
            id='a8ba390d-7a7e-4c40-99bc-4fddd56c22c3',
            tool_call_id='call_xcg6VNPMQbSv3QWLy2pdGOpu'
        ),
        ToolMessage(
            content='Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) 
when only one is expected.\n Please fix your mistakes.',
            name='EventDetails',
            id='a588be48-a558-49ad-af72-ba0b98cfed06',
            tool_call_id='call_hG46LFbb0mFpsLF7u82v9fGB'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_details': [
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqaM15r0U_liK19oZ0

In [58]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=False
        #handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"}]
})
# rprint(result)

for msg in result["messages"]:
    msg.pretty_print()

MultipleStructuredOutputsError: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.

In [60]:
agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors="请检查输入数据"
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"}]
})
rprint(result)

for msg in result["messages"]:
    msg.pretty_print()

{
    'messages': [
        HumanMessage(
            content='请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：
2026-07-15',
            additional_kwargs={},
            response_metadata={},
            id='ad9bfb77-4d04-4610-b13f-db3224dea442'
        ),
        AIMessage(
            content='',
            additional_kwargs={
                'reasoning_content': '**Considering tool usage**\n\nThe user wants me to extract two categories, so
I need to use multiple tools in parallel. I think I should just call them directly without adding any unnecessary 
preamble. It’s straightforward; I’ll focus on making sure the tools are invoked properly to get the desired 
categories efficiently. My priority is to be quick and effective in delivering what the user needs. Let’s get this 
done!',
                'reasoning_details': [
                    {
                        'type': 'reasoning.summary',
                        'summary': '**Considering tool usage**\n\nThe user wants me to extract two categories, so I
need to use multiple tools in parallel. I think I should just call them directly without adding any unnecessary 
preamble. It’s straightforward; I’ll focus on making sure the tools are invoked properly to get the desired 
categories efficiently. My priority is to be quick and effective in delivering what the user needs. Let’s get this 
done!',
                        'format': 'openai-responses-v1',
                        'index': 0.0
                    },
                    {
                        'type': 'reasoning.encrypted',
                        'data': 
'gAAAAABqaM6l4qdhA--r2qE-I1Zo_nz2-Ewpr4CBQdsbBN1Jl0gbvyIL7fhhlGBi2K6yM-Mc0VOLhW2AziWJIUoMecQbYgtAXTWvtlwMKI458Dx5KU
zkQUM5BdqlCXje9e03V4Hd8Ci4y7UZ--wxgGQOVXWDwAfoO5CVmq9VHRoENyQQH7XD-4ltx2yYPhrrWJRHAFragXipNjjqrtECkuHPqhvMmBQyISZMY
-yPDL1PlFtc2MHY5kdu-jEOLUUr7cDK6OYLYr0xrPnR4kn6QSSfGA-harGQ9Ar2-IsVJztyV4Xe0awlAZYAvOYRp50xUidNwT594iyO86gL3b6xsJfx
lher0_wEHKxx4B7cVGKEPZgqiWuXNbI_W9wjfVRE-R632Vodu5yOeptvhz2Da3Zv6aDe4F90Hoxt9kFb3UfLeuipuvbtrfCCniu6Kr3-R8eeqecvrLO
i7dw0nRH8YdqZmORa8yRG9wMD1ZzVxKkOokdHmSXxPhKDdnC-mJExY4nxmBMu-PTfMhRUB4WJ43YMQ2LYNr3LQnyKlww2xRQIJHNBa_R9sO2BPjcD5K
p9qd5ALh40Yj_eJOYW86PmWhTCvuPoQ_AM6mMqSrbLvndvMR8e1eQNYNCmFyTiQEBfTz35VLiCbwudVYKofFW4DrgYD15FrnIDnR48yFGcChA36Y2bA
Liy0djPbko49qFkiUbrpWo4oR0xkv8N7FrDfo4SVX-3NdKuj6-RlFIN5kgyEgSu2lvwq79en3qyLbWG6BAB0kVo4ebr5zJ7dSTYSHywjwWaL9WdduJ-
oJJAE_LpNXyYfJcB9fzRIYbb8rgDLQGYOrJAddn3GVnq_C3-wVYRkXh1eGR2gJHg7xHLjeUheJ7kFTMXblWNVjvCU4SNzfYhmgA9hCFNldUn-jxhMdL
RCkiQKjv8jVLZt32W3RaI5Q4q93_dKkBUZc1b2INoreffTSF6kzSoswNYwc_GPDG90NQCsxQNcDH62Nqkl2enEaEJhH5g1RN6jatPEYbVKOkxhMT_GD
Hvr_xC6F94n9EbGe94SuwtWB4PgJgg64YESiQOI8Gldjl8jwgnfMqNIOAkudE9FCM8',
                        'id': 'rs_0adc5ce5f72c6867016a68cea45ca0819fbb12a22c7b202f77',
                        'format': 'openai-responses-v1',
                        'index': 1.0
                    }
                ]
            },
            response_metadata={
                'model_name': 'openai/gpt-5.6-luna',
                'id': 'gen-1785253539-1hhAhMdCvtzaq5VpGnZn',
                'created': 1785253539,
                'object': 'chat.completion',
                'finish_reason': 'tool_calls',
                'logprobs': None,
                'model_provider': 'openrouter'
            },
            id='lc_run--019fa967-2e25-7b03-a7c2-3448712ec827-0',
            tool_calls=[
                {
                    'name': 'ContactInfo',
                    'args': {'name': '张三', 'email': 'zhang3@atguigu.com'},
                    'id': 'call_8u6OSpaT2CD6bPbhZWiUDt5m',
                    'type': 'tool_call'
                },
                {
                    'name': 'EventDetails',
                    'args': {'event_name': '公司年会', 'date': '2026-07-15'},
                    'id': 'call_Iw1md70AGXbDbfv28SgnmxSM',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metad

================================ Human Message =================================

请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_8u6OSpaT2CD6bPbhZWiUDt5m)
 Call ID: call_8u6OSpaT2CD6bPbhZWiUDt5m
  Args:
    name: 张三
    email: zhang3@atguigu.com
  EventDetails (call_Iw1md70AGXbDbfv28SgnmxSM)
 Call ID: call_Iw1md70AGXbDbfv28SgnmxSM
  Args:
    event_name: 公司年会
    date: 2026-07-15
================================= Tool Message =================================
Name: ContactInfo

请检查输入数据
================================= Tool Message =================================
Name: EventDetails

请检查输入数据
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_2KntOPXQs9QOvIEpJuGqeyoX)
 Call ID: call_2KntOPXQs9QOvIEpJuGqeyoX
  Args:
    name: 张三
    email: zhang3@atguigu.com
================================= Too

### 设置为指定异常类型

In [64]:
from langchain.agents.structured_output import MultipleStructuredOutputsError, StructuredOutputValidationError

agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=(MultipleStructuredOutputsError, StructuredOutputValidationError)
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"}]
})

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_7mjXJchdcUgc93Jv4KMZLiLr)
 Call ID: call_7mjXJchdcUgc93Jv4KMZLiLr
  Args:
    name: 张三
    email: zhang3@atguigu.com
  EventDetails (call_DKRL0GbLlHJP1mtbAz9Pi9Hy)
 Call ID: call_DKRL0GbLlHJP1mtbAz9Pi9Hy
  Args:
    event_name: 公司年会
    date: 2026-07-15
================================= Tool Message =================================
Name: ContactInfo

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
 Please fix your mistakes.
================================= Tool Message =================================
Name: EventDetails

Error: Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
 Please fix your m

### 设置为自定义错误处理函数

In [65]:
from langchain.agents.structured_output import MultipleStructuredOutputsError, StructuredOutputValidationError


# 自定义错误处理函数
def custom_error_handler(error: Exception) -> str:
    """自定义错误处理器"""
    error_str = str(error)
    print(f"捕获到错误类型：{type(error).__name__}")
    print(f"错误详情：{error_str}")
    if isinstance(error, StructuredOutputValidationError):
        return "数据格式有误，请检查字段是否符合要求。"
    elif isinstance(error, MultipleStructuredOutputsError):
        return "检测到多个响应，请选择最相关的一个进行返回。"
    else:
        return f"Error: {error_str}"


agent = create_agent(
    model=model,
    response_format=ToolStrategy(
        Union[ContactInfo, EventDetails],
        tool_message_content="提取完成！",
        handle_errors=custom_error_handler
    )
)

result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15"}]
})

for msg in result["messages"]:
    msg.pretty_print()

捕获到错误类型：MultipleStructuredOutputsError
错误详情：Model incorrectly returned multiple structured responses (ContactInfo, EventDetails) when only one is expected.
================================ Human Message =================================

请提取以下文本中内容：姓名：张三，电子邮箱：zhang3@atguigu.com，活动名称：公司年会，活动日期：2026-07-15
================================== Ai Message ==================================
Tool Calls:
  ContactInfo (call_84wmCpF1dRYQBAQwP9xd7yJ1)
 Call ID: call_84wmCpF1dRYQBAQwP9xd7yJ1
  Args:
    name: 张三
    email: zhang3@atguigu.com
  EventDetails (call_B2XUsQnzy8xPiHxRxQUwK7xP)
 Call ID: call_B2XUsQnzy8xPiHxRxQUwK7xP
  Args:
    event_name: 公司年会
    date: 2026-07-15
================================= Tool Message =================================
Name: ContactInfo

检测到多个响应，请选择最相关的一个进行返回。
================================= Tool Message =================================
Name: EventDetails

检测到多个响应，请选择最相关的一个进行返回。
================================== Ai Message ==================================
